# Use cross-validation to ensure model generalization by manually splitting data into k-folds, training and testing models iteratively without using built-in cross-validation functions, and computing aggregated performance metrics.

In [2]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


# 1. LOAD DATASET
df = pd.read_csv("heart.csv")

X = df.drop(columns=["HeartDisease"]).copy()
y = df["HeartDisease"].astype(int).values


# 2. HANDLE MISSING VALUES
for col in X.columns:
    if pd.api.types.is_numeric_dtype(X[col]):
        X[col] = X[col].fillna(X[col].median())
    else:
        mode = X[col].mode()
        X[col] = X[col].fillna(
            mode.iloc[0] if not mode.empty else "Unknown"
        ).astype(str)

X = pd.get_dummies(X, dtype=float)
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
X = X.to_numpy(dtype=float)


# 3. MANUAL K-FOLD SPLITTING
def manual_kfold(y, k=5, seed=42):
    rng = np.random.default_rng(seed)

    folds = [[] for _ in range(k)]

    # Split each class across folds to maintain class balance
    for cls in np.unique(y):
        indices = np.where(y == cls)[0].copy()
        rng.shuffle(indices)

        for i, idx in enumerate(indices):
            folds[i % k].append(idx)

    all_indices = np.arange(len(y))
    result = []

    for i in range(k):
        test_idx = np.array(folds[i], dtype=int)
        train_idx = np.setdiff1d(all_indices, test_idx)

        result.append((train_idx, test_idx))

    return result


# 4. MANUAL XGBOOST-STYLE MODEL FUNCTIONS
def sigmoid(z):
    z = np.clip(z, -35, 35)
    return 1 / (1 + np.exp(-z))


def find_best_stump(X, gradient, hessian):
    best_gain = 0
    best_stump = None
    reg_lambda = 1.0

    total_g = gradient.sum()
    total_h = hessian.sum()

    for feature in range(X.shape[1]):
        values = X[:, feature]
        unique = np.unique(values)

        if len(unique) < 2:
            continue

        if len(unique) > 12:
            thresholds = np.unique(
                np.quantile(values, np.linspace(0.1, 0.9, 9))
            )
        else:
            thresholds = (unique[:-1] + unique[1:]) / 2

        for threshold in thresholds:
            left = values <= threshold
            right = ~left

            if left.sum() < 5 or right.sum() < 5:
                continue

            gl, gr = gradient[left].sum(), gradient[right].sum()
            hl, hr = hessian[left].sum(), hessian[right].sum()

            gain = 0.5 * (
                gl**2 / (hl + reg_lambda)
                + gr**2 / (hr + reg_lambda)
                - total_g**2 / (total_h + reg_lambda)
            )

            if gain > best_gain:
                best_gain = gain
                best_stump = {
                    "feature": feature,
                    "threshold": threshold,
                    "left_value": -gl / (hl + reg_lambda),
                    "right_value": -gr / (hr + reg_lambda)
                }

    return best_stump


def train_manual_xgb(X, y, rounds=30, learning_rate=0.1):
    rate = np.clip(y.mean(), 1e-6, 1 - 1e-6)
    base_score = np.log(rate / (1 - rate))
    margin = np.full(len(y), base_score, dtype=float)
    learners = []

    for _ in range(rounds):
        probability = sigmoid(margin)
        gradient = probability - y
        hessian = np.maximum(probability * (1 - probability), 1e-6)

        stump = find_best_stump(X, gradient, hessian)

        if stump is None:
            break

        learners.append(stump)

        left = X[:, stump["feature"]] <= stump["threshold"]
        right = ~left

        margin[left] += learning_rate * stump["left_value"]
        margin[right] += learning_rate * stump["right_value"]

    return {
        "base_score": base_score,
        "learners": learners,
        "learning_rate": learning_rate
    }


def predict_manual_xgb(model, X):
    margin = np.full(len(X), model["base_score"], dtype=float)

    for stump in model["learners"]:
        left = X[:, stump["feature"]] <= stump["threshold"]
        right = ~left

        margin[left] += model["learning_rate"] * stump["left_value"]
        margin[right] += model["learning_rate"] * stump["right_value"]

    return (sigmoid(margin) >= 0.5).astype(int)


# 5. CALCULATE PERFORMANCE METRICS
def get_metrics(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1 Score": f1_score(y_true, y_pred, zero_division=0)
    }


# 6. MANUAL 5-FOLD CROSS-VALIDATION
k = 5
folds = manual_kfold(y, k=k)

results = []

for fold_no, (train_idx, test_idx) in enumerate(folds, start=1):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    print(f"\n========== FOLD {fold_no} ==========")

    # Random Forest
    rf = RandomForestClassifier(
        n_estimators=100,
        random_state=42
    )
    rf.fit(X_train, y_train)
    rf_pred = rf.predict(X_test)

    rf_metrics = get_metrics(y_test, rf_pred)
    results.append({"Fold": fold_no, "Model": "Random Forest", **rf_metrics})

    # Manual XGBoost-style Gradient Boosting
    xgb_model = train_manual_xgb(X_train, y_train)
    xgb_pred = predict_manual_xgb(xgb_model, X_test)

    xgb_metrics = get_metrics(y_test, xgb_pred)
    results.append({
        "Fold": fold_no,
        "Model": "Manual XGBoost-Style",
        **xgb_metrics
    })

    print("Random Forest:", rf_metrics)
    print("Manual XGBoost-Style:", xgb_metrics)


# 7. AGGREGATE PERFORMANCE METRICS
results_df = pd.DataFrame(results)

print("\n========== RESULTS FOR EACH FOLD ==========")
print(results_df.round(4).to_string(index=False))

summary = (
    results_df.groupby("Model")[
        ["Accuracy", "Precision", "Recall", "F1 Score"]
    ]
    .agg(["mean", "std"])
)

print("\n========== AGGREGATED CROSS-VALIDATION RESULTS ==========")
print(summary.round(4).to_string())




========== FOLD 1 ==========
Random Forest: {'Accuracy': 0.8913043478260869, 'Precision': 0.9183673469387755, 'Recall': 0.8823529411764706, 'F1 Score': 0.9}
Manual XGBoost-Style: {'Accuracy': 0.8913043478260869, 'Precision': 0.9019607843137255, 'Recall': 0.9019607843137255, 'F1 Score': 0.9019607843137255}

========== FOLD 2 ==========
Random Forest: {'Accuracy': 0.8858695652173914, 'Precision': 0.8715596330275229, 'Recall': 0.9313725490196079, 'F1 Score': 0.9004739336492891}
Manual XGBoost-Style: {'Accuracy': 0.8586956521739131, 'Precision': 0.8333333333333334, 'Recall': 0.9313725490196079, 'F1 Score': 0.8796296296296297}

========== FOLD 3 ==========
Random Forest: {'Accuracy': 0.8858695652173914, 'Precision': 0.8932038834951457, 'Recall': 0.9019607843137255, 'F1 Score': 0.8975609756097561}
Manual XGBoost-Style: {'Accuracy': 0.8695652173913043, 'Precision': 0.8545454545454545, 'Recall': 0.9215686274509803, 'F1 Score': 0.8867924528301887}

========== FOLD 4 ==========
Random Forest: {